In [24]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import MinMaxScaler

# Cargar y preprocesar el dataset
df = pd.read_csv("datos_reales.csv")  # Ruta correcta al dataset
df_clean = df.drop(columns=['Unnamed: 0'], errors='ignore')  # Ignorar si la columna no existe
#scaler = MinMaxScaler()
#X_scaled = scaler.fit_transform(df_clean.values)
scaler = MinMaxScaler(feature_range=(0, 1)) # Asegura que esté entre 0 y 1
X_scaled = scaler.fit_transform(df_clean.values)

print(X_scaled.shape)  # Verifica la forma de tus datos escalados

print("############################")  # Verifica la forma de tus datos escalados

# Definir parámetros
input_shape = X_scaled.shape[1]  # Número de características (12 en tu dataset)
latent_dim = 2  # Dimensión del espacio latente, simplificado

# Crear el encoder
def create_encoder(input_shape, latent_dim):
    inputs = layers.Input(shape=(input_shape,))
    x = layers.Dense(32, activation='relu')(inputs)
    x = layers.Dense(16, activation='relu')(x)
    z_mean = layers.Dense(latent_dim)(x)
    z_log_var = layers.Dense(latent_dim)(x)

    def sampling(args):
        z_mean, z_log_var = args
        epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

    z = layers.Lambda(sampling)([z_mean, z_log_var])
    return Model(inputs, [z_mean, z_log_var, z], name="encoder")

# Crear el decoder
def create_decoder(latent_dim, output_shape):
    latent_inputs = layers.Input(shape=(latent_dim,))
    x = layers.Dense(16, activation='relu')(latent_inputs)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(output_shape, activation='sigmoid')(x)  # Mantener la salida en el rango (0, 1)
    return Model(latent_inputs, outputs, name="decoder")

# Definir el modelo VAE
# class VAE(Model):
#     def __init__(self, encoder, decoder, beta=1):
#         super(VAE, self).__init__()
#         self.encoder = encoder
#         self.decoder = decoder
#         self.beta = beta

#     def call(self, inputs):
#         z_mean, z_log_var, z = self.encoder(inputs)
#         reconstruction = self.decoder(z)
#         reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(inputs, reconstruction), axis=1))
#         kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))
#         total_loss = reconstruction_loss + self.beta * kl_loss
#         self.add_loss(total_loss)
#         return reconstruction
# class VAE(Model):
#     def __init__(self, encoder, decoder, beta=1):
#         super(VAE, self).__init__()
#         self.encoder = encoder
#         self.decoder = decoder
#         self.beta = beta

#     def call(self, inputs):
#         z_mean, z_log_var, z = self.encoder(inputs)
#         reconstruction = self.decoder(z)
#         reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(inputs, reconstruction), axis=1))
#         kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1))
#         total_loss = reconstruction_loss + self.beta * kl_loss
#         self.add_loss(total_loss)
#         return reconstruction
class VAE(Model):
    def __init__(self, encoder, decoder):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        # Elimina la pérdida KL temporalmente para simplificar la depuración
        reconstruction_loss = tf.reduce_mean(tf.reduce_sum(tf.keras.losses.binary_crossentropy(inputs, reconstruction), axis=1))
        self.add_loss(reconstruction_loss)  # Solo usamos la pérdida de reconstrucción
        return reconstruction

# Crear el VAE
encoder = create_encoder(input_shape, latent_dim)
decoder = create_decoder(latent_dim, input_shape)
#vae = VAE(encoder, decoder, beta=1) saco beta temporalmente para ir viendo que pasa sin la funcion kl
vae = VAE(encoder, decoder)

# Compilar el modelo
vae.compile(optimizer='adam')
###############################
# Comprobar las formas para ver que pasa!
sample_input = tf.random.normal((1, input_shape))  # Crea un tensor de muestra con la forma de entrada
z_mean, z_log_var, z = encoder(sample_input)  # Pasa la muestra al encoder
print("Forma de z_mean:", z_mean.shape)
print("Forma de z_log_var:", z_log_var.shape)
print("Forma de z:", z.shape)

sample_output = decoder(z)  # Pasa z al decoder
print("Forma de salida del decoder:", sample_output.shape)
########################################
# Verificar la estructura del modelo
encoder.summary()
decoder.summary()

# Entrenar el modelo con los datos escalados
vae.fit(X_scaled, X_scaled, epochs=20, batch_size=8) #epochs=50 batch_size=16

# Generar nuevos datos sintéticos
def generate_synthetic_data(vae, num_samples=10000):
    # Muestrear datos del espacio latente
    z_sample = np.random.normal(size=(num_samples, latent_dim))
    # Decodificar los datos para generar datos sintéticos
    synthetic_data = vae.decoder.predict(z_sample)
    # Desescalar los datos a su rango original
    return scaler.inverse_transform(synthetic_data)

# Generar 10,000 datos sintéticos
synthetic_data = generate_synthetic_data(vae, num_samples=10000)

# Guardar los datos sintéticos en un archivo CSV
synthetic_df = pd.DataFrame(synthetic_data, columns=df_clean.columns)
synthetic_df.to_csv("datos_sinteticos.csv", index=False)

print("10,000 datos sintéticos generados y guardados en 'datos_sinteticos.csv'.")


(168, 12)
############################
Forma de z_mean: (1, 2)
Forma de z_log_var: (1, 2)
Forma de z: (1, 2)
Forma de salida del decoder: (1, 12)
Model: "encoder"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_41 (InputLayer)       [(None, 12)]                 0         []                            
                                                                                                  
 dense_140 (Dense)           (None, 32)                   416       ['input_41[0][0]']            
                                                                                                  
 dense_141 (Dense)           (None, 16)                   528       ['dense_140[0][0]']           
                                                                                                  
 dense_142 (Dense)           (None, 2)       

ValueError: in user code:

    File "/home/caipy/dev/projects/proyecto_ica_copia/.venv/lib/python3.8/site-packages/keras/src/engine/training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "/home/caipy/dev/projects/proyecto_ica_copia/.venv/lib/python3.8/site-packages/keras/src/engine/training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/caipy/dev/projects/proyecto_ica_copia/.venv/lib/python3.8/site-packages/keras/src/engine/training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "/home/caipy/dev/projects/proyecto_ica_copia/.venv/lib/python3.8/site-packages/keras/src/engine/training.py", line 1080, in train_step
        y_pred = self(x, training=True)
    File "/home/caipy/dev/projects/proyecto_ica_copia/.venv/lib/python3.8/site-packages/keras/src/utils/traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(f